# Assignment
## **Employee dataset creation & LLM prompting for Text to SQL generator.**

### Data Preparation via Mockaroo

In [ ]:
! curl "https://api.mockaroo.com/api/7ceee420?count=1000&key=869f6b20" > "Department.csv"

In [ ]:
! curl "https://api.mockaroo.com/api/e8d9bb20?count=1000&key=869f6b20" > "Position.csv"

In [ ]:
! curl "https://api.mockaroo.com/api/df73cff0?count=1000&key=869f6b20" > "Employees.csv"

In [ ]:
! curl "https://api.mockaroo.com/api/38fdcf50?count=1000&key=869f6b20" > "Audit.csv"

In [ ]:
! curl "https://api.mockaroo.com/api/09c68950?count=1000&key=869f6b20" > "Leave.csv"

In [ ]:
! curl "https://api.mockaroo.com/api/1c7a5490?count=1000&key=869f6b20" > "Performance.csv"

In [ ]:
! curl "https://api.mockaroo.com/api/2aa6a220?count=1000&key=869f6b20" > "Project.csv"

In [ ]:
! curl "https://api.mockaroo.com/api/f47360b0?count=1000&key=869f6b20" > "Salary.csv"

### Setup DB

In [ ]:
import sqlite3
import pandas as pd
import os

In [ ]:
department_schema = """CREATE TABLE IF NOT EXISTS department (
	department_id INT,
	department_name VARCHAR(50),
	location VARCHAR(50)
);
"""
poition_schema = """CREATE TABLE IF NOT EXISTS position (
	position_id INT,
	position_title VARCHAR(50),
	description TEXT,
	salary_range_min INT,
	salary_range_max INT
);
"""
employee_schema = """CREATE TABLE IF NOT EXISTS employee (
	employee_id INT,
	first_name VARCHAR(50),
	last_name VARCHAR(50),
	dob DATE,
	gender VARCHAR(50),
	email VARCHAR(50),
	phone_number VARCHAR(50),
	address VARCHAR(50),
	hire_date DATE,
	salary INT,
	position_id VARCHAR(50),
	department_id VARCHAR(50),
	manager_id VARCHAR(50),
	status VARCHAR(8)
);
"""
audit_schema="""CREATE TABLE IF NOT EXISTS audit (
	audit_id INT,
	employee_id VARCHAR(50),
	action_type VARCHAR(7),
	field_changed VARCHAR(10),
	old_value INT,
	new_value INT,
	timestamp DATE,
	performed_by VARCHAR(50)
);
"""
leave_schema="""CREATE TABLE IF NOT EXISTS leave (
	leave_id INT,
	employee_id VARCHAR(50),
	leave_type VARCHAR(15),
	start_date DATE,
	end_date DATE,
	status VARCHAR(9),
	approval_date DATE
);
"""
performance_schema="""CREATE TABLE IF NOT EXISTS performance (
	performance_id INT,
	employee_id VARCHAR(50),
	review_date DATE,
	rating VARCHAR(9),
	comments TEXT,
	goals_set TEXT
);
"""
project_schema="""CREATE TABLE IF NOT EXISTS project (
	project_id INT,
	project_name VARCHAR(50),
	description TEXT,
	start_date DATE,
	end_date DATE,
	project_manager_id VARCHAR(50),
	status VARCHAR(9)
);
"""
salary_schema="""CREATE TABLE IF NOT EXISTS salary (
	salary_id INT,
	employee_id VARCHAR(50),
	salary_amount DECIMAL(9,2),
	effective_date DATE,
	salary_type VARCHAR(10)
);
"""

In [ ]:
db_name = 'employee.db'
if os.path.exists(db_name):
    os.remove(db_name)
    print(f"Removed existing database '{db_name}'.")

In [ ]:
COLUMN_DATA_TYPES = {
    'department': {
        'department_id': 'int64',
        'department_name': 'object',
        'location': 'object'
    },
    'position': {
        'position_id': 'int64',
        'position_title': 'object',
        'description': 'object',
        'salary_range_min': 'float64',
        'salary_range_max': 'float64'
    },
    'employee': {
        'employee_id': 'int64',
        'first_name': 'object',
        'last_name': 'object',
        'dob': 'datetime64[ns]',
        'gender': 'object',
        'email': 'object',
        'phone_number': 'object',
        'address': 'object',
        'hire_date': 'datetime64[ns]',
        'salary': 'float64',
        'position_id': 'int64',
        'department_id': 'int64',
        'manager_id': 'int64',
        'status': 'object'
    },
    'salary': {
        'salary_id': 'int64',
        'employee_id': 'int64',
        'salary_amount': 'float64',
        'effective_date': 'datetime64[ns]',
        'salary_type': 'object'
    },
    'leave': {
        'leave_id': 'int64',
        'employee_id': 'int64',
        'leave_type': 'object',
        'start_date': 'datetime64[ns]',
        'end_date': 'datetime64[ns]',
        'status': 'object',
        'approval_date': 'datetime64[ns]'
    },
    'performance': {
        'performance_id': 'int64',
        'employee_id': 'int64',
        'review_date': 'datetime64[ns]',
        'rating': 'object',
        'comments': 'object',
        'goals_set': 'object'
    },
    'project': {
        'project_id': 'int64',
        'project_name': 'object',
        'description': 'object',
        'start_date': 'datetime64[ns]',
        'end_date': 'datetime64[ns]',
        'project_manager_id': 'int64',
        'status': 'object'
    },
    'audit': {
        'audit_id': 'int64',
        'employee_id': 'int64',
        'action_type': 'object',
        'field_changed': 'object',
        'old_value': 'object',
        'new_value': 'object',
        'timestamp': 'datetime64[ns]',
        'performed_by': 'object'
    }
}

db_name = 'employee.db'
conn = None
try:
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    print(f"Database '{db_name}' created and connected successfully.")

    cursor.execute(department_schema)
    cursor.execute(poition_schema)
    cursor.execute(employee_schema)
    cursor.execute(audit_schema)
    cursor.execute(leave_schema)
    cursor.execute(performance_schema)
    cursor.execute(project_schema)
    cursor.execute(salary_schema)
    print("Tables created successfully.")

    csv_to_table_map = {
        '/content/Department.csv': 'department',
        '/content/Position.csv': 'position',
        '/content/Employees.csv': 'employee',
        '/content/Audit.csv': 'audit',
        '/content/Leave.csv': 'leave',
        '/content/Performance.csv': 'performance',
        '/content/Project.csv': 'project',
        '/content/Salary.csv': 'salary'
    }

    for csv_file, table_name in csv_to_table_map.items():
        if os.path.exists(csv_file):
            print(f"\nProcessing '{csv_file}' for table '{table_name}'...")

            # Read the CSV file into a pandas DataFrame
            df = pd.read_csv(csv_file)

            # 1. Get the expected schema for the current table
            expected_schema = COLUMN_DATA_TYPES[table_name]
            expected_cols = list(expected_schema.keys())

            # 2. Handle missing/extra columns
            # Drop columns from DataFrame that are not in the schema
            df = df[df.columns.intersection(expected_cols)]

            # Add any missing columns and fill with None (which becomes NULL in SQL)
            for col in expected_cols:
                if col not in df.columns:
                    df[col] = None

            # 3. Reorder columns to match the defined schema exactly
            df = df[expected_cols]

            # 4. Enforce data types
            for col, dtype in expected_schema.items():
                if 'datetime' in dtype:
                    # Use pd.to_datetime for date/time columns, coercing errors to NaT (Not a Time)
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                else:
                    # Use astype for other columns, handling potential conversion errors
                    try:
                        df[col] = df[col].astype(dtype)
                    except (ValueError, TypeError) as e:
                        print(f"  - Warning: Could not convert column '{col}' to {dtype}. Error: {e}. Leaving as is.")


            # Use the to_sql method to insert the cleaned DataFrame
            df.to_sql(table_name, conn, if_exists='append', index=False)
            print(f"  -> Data from '{csv_file}' loaded into '{table_name}' table successfully.")
        else:
            print(f"Warning: '{csv_file}' not found. Skipping data load for '{table_name}'.")

    # Commit the changes to the database
    conn.commit()
    print("\nData committed to the database successfully. 🎉")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except pd.errors.EmptyDataError as e:
    print(f"Pandas error: {e}. One of the CSV files might be empty.")
except KeyError as e:
    print(f"Schema definition error: A column is missing from the TABLE_DATA_TYPES dictionary: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    # Close the connection if it was established
    if conn:
        conn.close()
        print("Database connection closed.")


### Google GenAI setup

In [ ]:
! pip install google-genai

In [ ]:
from google import genai
from google.colab import userdata
genai_client = genai.Client(api_key=userdata.get('GOOGLE_GENAI_API'))

### Prompt Engineering

In [ ]:
prompt = """
### **ROLE**

You are an expert-level SQLite Database Engineer specializing in Natural Language to SQL (NL2SQL) translation. Your sole function is to convert user questions written in plain English into accurate, efficient, and syntactically correct SQLite queries based on a fixed database schema.

-----

### **CONTEXT**

You are the core translation engine for an HR Analytics and Employee Management dashboard. This tool allows non-technical employees to query the company's HR database using natural language. The database dialect is always **SQLite**. Your responses will be executed directly on the database.

The database consists of the following tables:

**`department` table:**
```sql
CREATE TABLE department (
    department_id INT,
    department_name VARCHAR(50),
    location VARCHAR(50)
);
````

**`position` table:**

```sql
CREATE TABLE position (
    position_id INT,
    position_title VARCHAR(50),
    description TEXT,
    salary_range_min INT,
    salary_range_max INT
);
```

**`employee` table:**

```sql
CREATE TABLE employee (
    employee_id INT,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    dob DATE,
    gender VARCHAR(50),
    email VARCHAR(50),
    phone_number VARCHAR(50),
    address VARCHAR(50),
    hire_date DATE,
    salary INT,
    position_id INT,
    department_id INT,
    manager_id INT,
    status VARCHAR(8)
);
```

**`salary` table:**

```sql
CREATE TABLE salary (
    salary_id INT,
    employee_id INT,
    salary_amount DECIMAL(9,2),
    effective_date DATE,
    salary_type VARCHAR(10)
);
```

**`leave` table:**

```sql
CREATE TABLE leave (
    leave_id INT,
    employee_id INT,
    leave_type VARCHAR(15),
    start_date DATE,
    end_date DATE,
    status VARCHAR(9),
    approval_date DATE
);
```

**`performance` table:**

```sql
CREATE TABLE performance (
    performance_id INT,
    employee_id INT,
    review_date DATE,
    rating VARCHAR(9),
    comments TEXT,
    goals_set TEXT
);
```

**`project` table:**

```sql
CREATE TABLE project (
    project_id INT,
    project_name VARCHAR(50),
    description TEXT,
    start_date DATE,
    end_date DATE,
    project_manager_id INT,
    status VARCHAR(9)
);
```

**`audit` table:**

```sql
CREATE TABLE audit (
    audit_id INT,
    employee_id INT,
    action_type VARCHAR(7),
    field_changed VARCHAR(10),
    old_value INT,
    new_value INT,
    timestamp DATE,
    performed_by VARCHAR(50)
);
```

---

### **TASK**

Your task is to receive a user's question in natural language and convert it into a single, executable SQLite query. Follow these steps meticulously:

1. **Analyze the User's Query:** Understand the intent, required fields, conditions, and aggregations (`SUM`, `COUNT`, `AVG`, etc.).
2. **Map to the Schema:** Use the correct tables (`employee`, `department`, `position`, etc.) and relationships (e.g., `employee.department_id = department.department_id`, `employee.position_id = position.position_id`).
3. **Construct the SQLite Query:** Ensure syntactically correct and efficient SQL for SQLite.
4. **Handle Ambiguity:** If vague, ask for clarification instead of guessing.
5. **Reject Impossible Queries:** If schema does not support the request, respond with an error.

---

### **CONSTRAINTS**

* **Read-Only Operations:** Only generate `SELECT` queries.
* **Schema Strictness:** Do not invent tables/columns.
* **No Explanations:** Output only the JSON with query/clarification/error.
* **Single Query Only:** Must return one complete SQL statement.
* **Impossibility:** If not answerable, return `"error"` with explanation.

---

### **EXAMPLES**

**Example 1: Simple Lookup**

* **User Query:** "List all employees in the Finance department"
* **Expected Output:**

```json
{
  "status": "success",
  "response": "SELECT e.first_name, e.last_name FROM employee e INNER JOIN department d ON e.department_id = d.department_id WHERE d.department_name = 'Finance';"
}
```

**Example 2: Aggregation**

* **User Query:** "What is the average salary of employees in IT?"
* **Expected Output:**

```json
{
  "status": "success",
  "response": "SELECT AVG(e.salary) FROM employee e INNER JOIN department d ON e.department_id = d.department_id WHERE d.department_name = 'IT';"
}
```

**Example 3: Join with Position**

* **User Query:** "Show me employees with their job titles and salaries"
* **Expected Output:**

```json
{
  "status": "success",
  "response": "SELECT e.first_name, e.last_name, p.position_title, e.salary FROM employee e INNER JOIN position p ON e.position_id = p.position_id;"
}
```

**Example 4: Leave Records**

* **User Query:** "Find employees currently on approved leave"
* **Expected Output:**

```json
{
  "status": "success",
  "response": "SELECT e.first_name, e.last_name, l.leave_type, l.start_date, l.end_date FROM leave l INNER JOIN employee e ON l.employee_id = e.employee_id WHERE l.status = 'Approved' AND date('now') BETWEEN l.start_date AND l.end_date;"
}
```

**Example 5: Project Assignment**

* **User Query:** "Which projects are managed by John Doe?"
* **Expected Output:**

```json
{
  "status": "success",
  "response": "SELECT p.project_name, p.start_date, p.end_date FROM project p INNER JOIN employee e ON p.project_manager_id = e.employee_id WHERE e.first_name = 'John' AND e.last_name = 'Doe';"
}
```

**Example 6: Ambiguous Query**

* **User Query:** "Show me employee performance"
* **Expected Output:**

```json
{
  "status": "clarification_needed",
  "response": "Could you please clarify what you mean by 'performance'? For example: recent performance reviews, average ratings by department, or employees with top ratings?"
}
```

**Example 7: Impossible Query**

* **User Query:** "Which branch has the highest sales revenue?"
* **Expected Output:**

```json
{
  "status": "error",
  "response": "I cannot answer this question as the database does not contain sales or branch revenue information."
}
```

---

### **OUTPUT FORMAT**

Your final response must be a single JSON object with two keys:

1. `"status"`: One of `"success"`, `"clarification_needed"`, or `"error"`.
2. `"response"`:

   * If `"success"`, this will be a valid SQLite `SELECT` query.
   * If `"clarification_needed"`, this will be a clarifying question.
   * If `"error"`, this will explain why the query cannot be generated.
     """


### Execution

In [ ]:
import json
def get_sql_query(genai_client, prompt, user_query):

  contents = f"""
  {prompt}

  Here's the user query in english you need to work on:
  {user_query}
  """
  response = genai_client.models.generate_content(model='gemini-2.5-flash', contents=contents)
  print("Raw Response from model starts\n")
  print(response)
  print("Raw Response from model ends\n")

  # Access the usage_metadata attribute
  usage_metadata = response.usage_metadata

  # Print the different token counts
  print(f"Input Token Count: {usage_metadata.prompt_token_count}")
  print(f"Thoughts Token Count: {response.usage_metadata.thoughts_token_count}")
  print(f"Output Token Count: {usage_metadata.candidates_token_count}")
  print(f"Total Token Count: {usage_metadata.total_token_count}")

  output = json.loads(response.text.replace('```json', '').replace('```', ''))

  return output

In [ ]:
def execute_query(query, db_name='employee.db'):

    conn = None
    try:
        # Connect to the database
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Execute the query
        print(f"\nExecuting query on '{db_name}':\n{query}")
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Get column names from the cursor description
        columns = [description[0] for description in cursor.description]

        # Format results as a dataframe for easier use
        results_as_dict = [dict(zip(columns, row)) for row in results]
        results_df = pd.DataFrame(results_as_dict)

        print("Query executed successfully.")
        return results_df

    except sqlite3.Error as e:
        print(f"Database error executing query: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    finally:
        if conn:
            conn.close()

In [ ]:
def text2sql(genai_client, prompt, user_query):
  output = get_sql_query(genai_client, prompt, user_query)
  if output['status'] == 'success':
    results = execute_query(output['response'])
    return results
  return output

### Text to SQL Query Execution

In [ ]:
text2sql(genai_client, prompt, "List all employees")

In [ ]:
text2sql(genai_client, prompt, "Show me all female employees hired after 2020")

In [ ]:
text2sql(genai_client, prompt, "Show the salary history for employee with ID 101")

In [ ]:
text2sql(genai_client, prompt, "Show employees with performance rating 'Excellent'")

In [ ]:
text2sql(genai_client, prompt, "Show the last 10 audit records for employee 105")